In [1]:
import requests
import bs4
from sys import argv
import re
from sys import stdout
from gensim.models import KeyedVectors
import gensim.downloader as api
from os.path import expanduser
from time import perf_counter
from functools import lru_cache

In [2]:
class IllegalArgumentError(ValueError):
    pass

MODEL_NAME = "word2vec-google-news-300"

In [4]:
stdout.write(f"⏳ Loading {MODEL_NAME}... ")
stdout.flush()
try:
    starting_time_model = perf_counter()
    model = KeyedVectors.load_word2vec_format(expanduser(f"~/gensim-data/{MODEL_NAME}/{MODEL_NAME}.gz"), binary=True)
    ending_time_model = perf_counter()
except Exception as e:
    stdout.write(f"\nError loading model: {e}\n")
    stdout.flush()
    model = api.load(MODEL_NAME)
print("Model loaded successfully. ✅")
print(f"(loaded from local file in {ending_time_model - starting_time_model:.2f} seconds)")

⏳ Loading word2vec-google-news-300... Model loaded successfully. ✅
(loaded from local file in 113.47 seconds)


In [5]:
@lru_cache()
def cosine_sim(title1, title2):
	avg_v1 = get_word_embedding(title1)
	avg_v2 = get_word_embedding(title2)
	if avg_v1 is None:
		return float('inf')
	return 1 - model.cosine_similarities(avg_v1, [avg_v2])[0]

def is_dest_embedding_possible(title):
	words = title.split("_")
	for word in words:
		if word in model:
			return True
		elif word.lower() in model:
			return True
	return False

def get_word_embedding(phrase):
	words = phrase.split("_")
	sum_v = [0.0] * model.vector_size
	n_words = 0
	for word in words:
		if word in model:
			sum_v += model.get_vector(word)
			n_words += 1
		elif word.lower() in model:
			sum_v += model.get_vector(word.lower())
			n_words += 1
	if n_words == 0:
		return None
	avg_v = sum_v / n_words
	return avg_v

In [18]:
def check_links(origin_link, dest_link):
    if re.match(r"https://[a-z]{2}.wikipedia.org/wiki/", origin_link) is None:
        raise IllegalArgumentError("Origin link is not a Wikipedia link! Valid wiki link example: https://<language>.wikipedia.org/wiki/<title>")
    if re.match(r"https://[a-z]{2}.wikipedia.org/wiki/", dest_link) is None:
        raise IllegalArgumentError("Destination link is not a Wikipedia link! Valid wiki link example: https://<language>.wikipedia.org/wiki/<title>")
    origin_title = origin_link.split("/wiki/")[1]
    dest_title = dest_link.split("/wiki/")[1]
    lang = origin_link.split("/")[2].split(".")[0]
    if lang != dest_link.split("/")[2].split(".")[0]:
        raise IllegalArgumentError("Origin and destination links must be in the same language Wikipedia!")
    print(f"GAME SETTINGS ⚙️\nOrigin: {origin_link}\nDestination: {dest_link}\nLanguage: {lang}\n" + "-"*120)
    session = requests.Session()
    session.headers.update({
        "User-Agent": "MyPythonWikipediaClient/1.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    })
    origin_page = session.get(origin_link)
    if origin_page.status_code != 200:
        raise IllegalArgumentError("Origin page does not exist!")
    dest_page = session.get(dest_link)
    if dest_page.status_code != 200:
        raise IllegalArgumentError("Destination page does not exist!")

In [21]:
def play(origin_link, dest_link):
    origin_title = origin_link.split("/wiki/")[1]
    dest_title = dest_link.split("/wiki/")[1]
    lang = origin_link.split("/")[2].split(".")[0]
    num_links = 1
    history = [origin_title]
    min_distance = cosine_sim(origin_title, dest_title)
    distances = [min_distance]
    found = False
    curr_link = origin_link
    starting_time = perf_counter()
    visited_links = set()
    visited_links.add(origin_link)
    session = requests.Session()
    session.headers.update({
		"User-Agent": "MyPythonWikipediaClient/1.0",
		"Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
	})
    while not found:
        print(f"CURRENT PAGE: {curr_link:70}\tDistance to target: {min_distance:.10f}")
        curr_page = session.get(curr_link)
        soup = bs4.BeautifulSoup(curr_page.text, 'html.parser')
        links = soup.find_all('a', href=True) # get every link in <a>
        found = False
        min_distance = float('inf')
        next_link = None
        for link in links:
            href = link['href']
            if href.startswith("/wiki/") and ':' not in href:
                full_link = f"https://{lang}.wikipedia.org{href}"
                if full_link in visited_links:
                    continue
                if full_link == dest_link:
                    history.append(dest_title)
                    distances.append(0.0)
                    print(f"CURRENT PAGE: {full_link:70}\tDistance to target: {0.0:.10f}\n" + "-"*120)
                    ending_time = perf_counter()
                    found = True
                    break
                curr_title = href.split("/wiki/")[1]
                distance = cosine_sim(curr_title, dest_title)
                if distance < min_distance:
                    min_distance = distance
                    next_link = full_link
        else:
            if next_link is None:
                raise Exception("No valid links found on the page! 💀")
            history.append(next_link.split("/wiki/")[1])
            visited_links.add(next_link)
            distances.append(min_distance)
            num_links += 1
            curr_link = next_link
    print("\n\nAI WINS! 🎉")
    print(f"Number of links traversed: {num_links}")
    print(f"Total time taken: {ending_time - starting_time:.3f} seconds")

In [22]:
origin_link = "https://en.wikipedia.org/wiki/Barack_Obama"
dest_link = "https://en.wikipedia.org/wiki/Artificial_intelligence"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Barack_Obama
Destination: https://en.wikipedia.org/wiki/Artificial_intelligence
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Barack_Obama                            	Distance to target: 0.9210387865
CURRENT PAGE: https://en.wikipedia.org/wiki/Central_Intelligence_Agency             	Distance to target: 0.5758052835
CURRENT PAGE: https://en.wikipedia.org/wiki/Open-source_intelligence                	Distance to target: 0.2878684242
CURRENT PAGE: https://en.wikipedia.org/wiki/All-source_intelligence                 	Distance to target: 0.2878684242
CURRENT PAGE: https://en.wikipedia.org/wiki/Military_intelligence                   	Distance to target: 0.3427541986
CURRENT PAGE: https://en.wikipedia.org/wiki/Artificial_intelligence_arms_race       	Distance to target: 0.2006176836
CURRENT PAGE: https://e

In [23]:
origin_link = "https://en.wikipedia.org/wiki/Julius_Caesar"
dest_link = "https://en.wikipedia.org/wiki/Quantum_computing"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Julius_Caesar
Destination: https://en.wikipedia.org/wiki/Quantum_computing
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Julius_Caesar                           	Distance to target: 0.9162896194
CURRENT PAGE: https://en.wikipedia.org/wiki/Temple_of_Jupiter_Optimus_Maximus       	Distance to target: 0.6647084015
CURRENT PAGE: https://en.wikipedia.org/wiki/Cornice                                 	Distance to target: 0.6844759946
CURRENT PAGE: https://en.wikipedia.org/wiki/Architecture                            	Distance to target: 0.6935821410
CURRENT PAGE: https://en.wikipedia.org/wiki/Design_computing                        	Distance to target: 0.3268308459
CURRENT PAGE: https://en.wikipedia.org/wiki/Spatial_computing                       	Distance to target: 0.2800512257
CURRENT PAGE: https://en.wik

In [24]:
origin_link = "https://en.wikipedia.org/wiki/Taylor_Swift"
dest_link = "https://en.wikipedia.org/wiki/Game_theory"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Taylor_Swift
Destination: https://en.wikipedia.org/wiki/Game_theory
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Taylor_Swift                            	Distance to target: 0.9343295696
CURRENT PAGE: https://en.wikipedia.org/wiki/End_Game_(song)                         	Distance to target: 0.3910139299
CURRENT PAGE: https://en.wikipedia.org/wiki/Snake_(video_game_genre)                	Distance to target: 0.5796189257
CURRENT PAGE: https://en.wikipedia.org/wiki/Roguelike_deck-building_game            	Distance to target: 0.5379901959
CURRENT PAGE: https://en.wikipedia.org/wiki/Shooter_game                            	Distance to target: 0.5380421890
CURRENT PAGE: https://en.wikipedia.org/wiki/Galaxy_Game                             	Distance to target: 0.3964886517
CURRENT PAGE: https://en.wikipedia.

In [25]:
origin_link = "https://en.wikipedia.org/wiki/Photosynthesis"
dest_link = "https://en.wikipedia.org/wiki/Pizza"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Photosynthesis
Destination: https://en.wikipedia.org/wiki/Pizza
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Photosynthesis                          	Distance to target: 0.9408726346
CURRENT PAGE: https://en.wikipedia.org/wiki/Nut_(fruit)                             	Distance to target: 0.6163731266
CURRENT PAGE: https://en.wikipedia.org/wiki/Snack                                   	Distance to target: 0.5101689811
CURRENT PAGE: https://en.wikipedia.org/wiki/Pizza                                   	Distance to target: 0.0000000000
------------------------------------------------------------------------------------------------------------------------


AI WINS! 🎉
Number of links traversed: 3
Total time taken: 1.392 seconds


In [26]:
origin_link = "https://en.wikipedia.org/wiki/Mount_Everest"
dest_link = "https://en.wikipedia.org/wiki/Marvel_Cinematic_Universe"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Mount_Everest
Destination: https://en.wikipedia.org/wiki/Marvel_Cinematic_Universe
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Mount_Everest                           	Distance to target: 0.8929345381
CURRENT PAGE: https://en.wikipedia.org/wiki/Snowman_Trek                            	Distance to target: 0.6091221162
CURRENT PAGE: https://en.wikipedia.org/wiki/Altitude_sickness                       	Distance to target: 0.8093565491
CURRENT PAGE: https://en.wikipedia.org/wiki/Google_Books                            	Distance to target: 0.6515734819
CURRENT PAGE: https://en.wikipedia.org/wiki/Grimoire                                	Distance to target: 0.5280050661
CURRENT PAGE: https://en.wikipedia.org/wiki/Shapeshifting                           	Distance to target: 0.4575548355
CURRENT PAGE: https:

In [27]:
origin_link = "https://en.wikipedia.org/wiki/Wolf"
dest_link = "https://en.wikipedia.org/wiki/Artificial_neural_network"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Wolf
Destination: https://en.wikipedia.org/wiki/Artificial_neural_network
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Wolf                                    	Distance to target: 0.8548010053
CURRENT PAGE: https://en.wikipedia.org/wiki/Phylogenetic                            	Distance to target: 0.5479170677
CURRENT PAGE: https://en.wikipedia.org/wiki/Phylogenetic_network                    	Distance to target: 0.3816456892
CURRENT PAGE: https://en.wikipedia.org/wiki/Molecular_phylogenetics                 	Distance to target: 0.5199095030
CURRENT PAGE: https://en.wikipedia.org/wiki/Inferring_horizontal_gene_transfer#Explicit_phylogenetic_methods	Distance to target: 0.4687436706
CURRENT PAGE: https://en.wikipedia.org/wiki/Horizontal_gene_transfer_in_evolution   	Distance to target: 0.5437539813
CURRE

In [28]:
origin_link = "https://en.wikipedia.org/wiki/Dolphin"
dest_link = "https://en.wikipedia.org/wiki/Internet"
check_links(origin_link, dest_link)
play(origin_link, dest_link)

GAME SETTINGS ⚙️
Origin: https://en.wikipedia.org/wiki/Dolphin
Destination: https://en.wikipedia.org/wiki/Internet
Language: en
------------------------------------------------------------------------------------------------------------------------
CURRENT PAGE: https://en.wikipedia.org/wiki/Dolphin                                 	Distance to target: 0.9701796363
CURRENT PAGE: https://en.wikipedia.org/wiki/YouTube                                 	Distance to target: 0.5176808096
CURRENT PAGE: https://en.wikipedia.org/wiki/Internet                                	Distance to target: 0.0000000000
------------------------------------------------------------------------------------------------------------------------


AI WINS! 🎉
Number of links traversed: 2
Total time taken: 1.593 seconds
